# Download Reasoning Traces (SYNTHETIC-1)
Streams `PrimeIntellect/SYNTHETIC-1` — 1.99M R1 reasoning traces across math, code, SWE, STEM Q&A.
Each example is reformatted as a chat-tokenized document using the special tokens reserved in our tokenizer:

```
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
<think>...</think>
{answer}<|im_end|>
```

(The `<think>...</think>` block is already inside `llm_response` from R1.)

Filtered on quality score. Resumable. Output ~30-50 GB into `data_reasoning/`.

In [ ]:
!pip install -q datasets

In [ ]:
import os
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/synapse/datasets_pretrain/data_reasoning"
os.makedirs(OUT_DIR, exist_ok=True)

DOCS_PER_FILE = 5_000
MIN_LENGTH = 100              # skip near-empty traces
MIN_SCORE = 0.5               # keep entries with score >= 0.5 OR no score recorded
MAX_DOCS = 100_000_000        # sentinel; exits when stream ends
SEPARATOR = "\n<|endoftext|>\n"
PREFIX = "reasoning_part"

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"

def format_example(prompt, response):
    return (
        f"{IM_START}user\n{prompt}{IM_END}\n"
        f"{IM_START}assistant\n{response}{IM_END}"
    )

# Resume detection
existing = sorted([
    f for f in os.listdir(OUT_DIR)
    if f.startswith(f"{PREFIX}_") and f.endswith(".txt")
])
if existing:
    last_num = int(existing[-1].replace(f"{PREFIX}_", "").replace(".txt", ""))
    docs_done = (last_num + 1) * DOCS_PER_FILE
    print(f"Found {len(existing)} existing files (~{docs_done:,} docs)")
    print(f"Resuming from doc {docs_done:,}...")
else:
    docs_done = 0

print(f"\nStreaming PrimeIntellect/SYNTHETIC-1 (1.99M traces, R1)...")
ds = load_dataset(
    "PrimeIntellect/SYNTHETIC-1",
    split="train",
    streaming=True,
)

buf = []
file_idx = len(existing)
total_chars = 0
total_docs = 0
skipped_short = 0
skipped_score = 0

for i, example in enumerate(ds):
    if i >= MAX_DOCS:
        break
    if i < docs_done:
        if i % 100_000 == 0 and i > 0:
            print(f"  Skipping... {i:,}/{docs_done:,}")
        continue

    # Score filter: include if no score recorded, exclude if score below threshold
    score = example.get("score")
    if score is not None and score < MIN_SCORE:
        skipped_score += 1
        continue

    prompt = (example.get("prompt") or "").strip()
    response = (example.get("llm_response") or "").strip()
    if len(prompt) < MIN_LENGTH or len(response) < MIN_LENGTH:
        skipped_short += 1
        continue

    doc = format_example(prompt, response)
    buf.append(doc)
    total_docs += 1

    if len(buf) >= DOCS_PER_FILE:
        path = os.path.join(OUT_DIR, f"{PREFIX}_{file_idx:04d}.txt")
        content = SEPARATOR.join(buf)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        size_mb = os.path.getsize(path) / 1024 / 1024
        total_chars += len(content)
        total_gb = total_chars / 1024 / 1024 / 1024
        print(f"  {path.split('/')[-1]}: {size_mb:.0f} MB | {total_docs:,} docs | {total_gb:.1f} GB | skipped {skipped_short:,} short / {skipped_score:,} low-score")
        buf = []
        file_idx += 1

# Write remainder
if buf:
    path = os.path.join(OUT_DIR, f"{PREFIX}_{file_idx:04d}.txt")
    content = SEPARATOR.join(buf)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    total_chars += len(content)
    file_idx += 1

total_gb = total_chars / 1024 / 1024 / 1024
print(f"\nDone!")
print(f"  {total_docs:,} reasoning traces in {file_idx} files, ~{total_gb:.1f} GB")
print(f"  Skipped: {skipped_short:,} short, {skipped_score:,} low-score")
print(f"  Saved to: {OUT_DIR}")
print(f"\nNext: run tokenizer_pipeline -> pre_train")